In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [2]:
from peft import PeftModel

finetuned_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

finetuned_model = PeftModel.from_pretrained(
    finetuned_model,
    "/kaggle/input/models/nikhil8885/lammaa-1b/pytorch/default/1/final_model"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [3]:
def generate_response(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [4]:
questions = [
    "मुझे बुखार और सिरदर्द है, क्या करूं?",
    "डायबिटीज के लक्षण क्या होते हैं?",
    "खांसी लंबे समय तक रहे तो क्या समस्या हो सकती है?",
    "ब्लड प्रेशर कैसे कंट्रोल करें?",
]

In [5]:
for q in questions:
    print("="*50)
    print("Question:", q)
    
    print("\n--- Base Model ---")
    print(generate_response(base_model, q))
    
    print("\n--- Fine-tuned Model ---")
    print(generate_response(finetuned_model, q))

Question: मुझे बुखार और सिरदर्द है, क्या करूं?

--- Base Model ---
मुझे बुखार और सिरदर्द है, क्या करूं? अगर इस आकर्षण में उनके साथ भी बताते हैं, तो फिर लिखें । इस मौके में हम हमारे भारतीय और अंतरराष्ट्रीय कल्याणकारी नौकरी के लिए भी स्थान

--- Fine-tuned Model ---
मुझे बुखार और सिरदर्द है, क्या करूं? आपका बुखार के फैले बुखार और सिरदर्द फैल जा रहा है। इसके कारण आपका इलाज क्या किया जा रहा है? आपका संक्रमण का एक हालत है, जो संसाधन रोग औ
Question: डायबिटीज के लक्षण क्या होते हैं?

--- Base Model ---
डायबिटीज के लक्षण क्या होते हैं?

Essay: The importance of cybersecurity lacks no doubt

उद्योग इलेक्ट्रॉनिकी की प्रतियोगिता की तरह से किस फाइनल में गए हैं?

Essay: The impact of IT industry on the nation's economy

--- Fine-tuned Model ---
डायबिटीज के लक्षण क्या होते हैं? जानें इन लक्षणों के बारे में क्या बताया गया है कि क्या आप चाहे यह हैं डायबिटीज साथ ही उसका इलाज करना चाहिए, जिससे कि क्या चाहे यह हैं डायबिटी
Question: खांसी लंबे समय तक रहे तो क्या समस्या हो सकती है?

--- Base Model ---
खांसी

In [6]:
questions = [
    "मुझे बुखार और सिरदर्द है, क्या करूं?",
    "डायबिटीज के लक्षण क्या होते हैं?",
    "खांसी लंबे समय तक रहे तो क्या समस्या हो सकती है?",
    "ब्लड प्रेशर कैसे कंट्रोल करें?",
]

In [7]:
import torch
import math

def compute_perplexity_text(model, tokenizer, texts):
    model.eval()
    losses = []

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss

        losses.append(loss.item())

    avg_loss = sum(losses) / len(losses)
    perplexity = math.exp(avg_loss)

    return perplexity

In [8]:
base_ppl = compute_perplexity_text(base_model, tokenizer, questions)
ft_ppl = compute_perplexity_text(finetuned_model, tokenizer, questions)

print("Base Model Perplexity:", round(base_ppl, 2))
print("Fine-tuned Model Perplexity:", round(ft_ppl, 2))

Base Model Perplexity: 5.79
Fine-tuned Model Perplexity: 2.76
